# 04 — Label Engineering

Applies the KDIGO AKI criteria to the engineered features to generate
**prospective** prediction targets, then produces the chronological
train/val/test splits used for modeling.

KDIGO Stage-1 AKI (evaluated on **both** channels now that we use MIMIC-IV):
- Serum creatinine increase ≥ 0.3 mg/dL above baseline, **OR** ≥ 1.5× baseline, **OR**
- Urine output < 0.5 mL/kg/h sustained for ≥ 6 h (mask-aware — missing hours are *not* read as anuria).

For each hour *t* we label whether **new-onset** AKI occurs within the next 6, 12, or 24 hours. Rows already in AKI, too close to discharge, or with an unobservable outcome are excluded (`valid_for_training`).

> **Order matters:** labeling happens *before* the split so the train/val/test
> parquets carry features **and** labels together. The split is chronological by
> patient (no leakage).

In [10]:
# Colab setup — mount Drive and cd into the project so ../data/... paths resolve.
# Safe anywhere: does nothing when not running on Colab.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import os, sys
    PROJ = '/content/drive/MyDrive/sentinel-poc/project_sentinel'
    os.chdir(os.path.join(PROJ, 'notebooks'))
    sys.path.insert(0, PROJ)
except ModuleNotFoundError:
    pass  # not on Colab — assume already running from notebooks/
import os
print('cwd =', os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cwd = /content/drive/MyDrive/sentinel-poc/project_sentinel/notebooks


In [11]:
import os
import sys
from pathlib import Path
import pandas as pd

sys.path.append('..')
from src import labels

interim = Path('../data/interim')
processed = Path('../data/processed')
processed.mkdir(parents=True, exist_ok=True)

# Raw cohort (un-imputed measurements) — labels are derived from THIS.
cohort_df = pd.read_parquet(interim / 'cohort_defined.parquet')
# Engineered features (imputed) — labels are attached to THIS for modeling.
features_df = pd.read_parquet(interim / 'engineered_features.parquet')
baseline_df = pd.read_parquet(interim / 'baseline_creatinine.parquet')

print(f"Raw cohort:          {cohort_df.shape}")
print(f"Engineered features: {features_df.shape}")
print(f"Baselines:           {baseline_df.shape}")

Raw cohort:          (8242612, 42)
Engineered features: (2707534, 203)
Baselines:           (92561, 2)


In [12]:
horizons = [6, 12, 24]

# Derive KDIGO labels from the RAW cohort (actual creatinine/urine measurements).
# Labeling on the imputed features would leak future values (ffill+bfill) into the label.
print("Applying prospective KDIGO horizon labels (creatinine + urine output)...")
labeled = labels.label_dataset(cohort_df, baseline_df, horizons=horizons)

# Attach the label columns onto the engineered features by (patient_id, hour).
label_cols = ['patient_id', 'ICULOS', '_aki_current', 'valid_for_training'] + \
             [f'aki_label_{h}h' for h in horizons]
labeled_df = features_df.merge(labeled[label_cols], on=['patient_id', 'ICULOS'], how='inner')

print(f"Labeled feature matrix: {labeled_df.shape}")
print(f"Valid-for-training rows: {int(labeled_df['valid_for_training'].sum()):,} "
      f"of {len(labeled_df):,}")

Applying prospective KDIGO horizon labels (creatinine + urine output)...
Labeled feature matrix: (2707534, 208)
Valid-for-training rows: 2,399,774 of 2,707,534


In [13]:
print("Label distribution (valid training rows only):")
labels.print_label_distribution(labeled_df, horizons)

Label distribution (valid training rows only):
6h:  10.5% positive, 89.5% negative (ratio: 8.5:1)
12h: 17.0% positive, 83.0% negative (ratio: 4.9:1)
24h: 23.4% positive, 76.6% negative (ratio: 3.3:1)


## Class Imbalance Discussion

As seen in the label distributions above, Sepsis-Associated AKI is a highly imbalanced outcome, typically occurring in a minority of clinical observations. Models trained on such imbalanced data tend to predict the majority class (non-AKI) at the expense of recall for the positive class (AKI).

To handle this during model training, we can employ strategies such as:
1. **`scale_pos_weight`**: Used in tree-based models like XGBoost and LightGBM to scale the gradient of the minority class proportionally.
2. **Class weights**: Specifying `class_weight='balanced'` in scikit-learn models like Logistic Regression or Random Forests to adjust weights inversely proportional to class frequencies.
3. **Threshold tuning**: Setting lower prediction thresholds optimizing for Area Under the Precision-Recall Curve (AUPRC) or F1-score rather than simple Accuracy.

In [ ]:
from src.features import temporal_split

# Train only on rows with a determinate, non-prevalent label.
valid_df = labeled_df[labeled_df['valid_for_training']].copy()

# Chronological split by patient (reserves any site 'B' as an external holdout).
splits = temporal_split(
    valid_df,
    val_ratio=0.15,
    test_ratio=0.15,
    output_dir='../data/processed',
    aki_label_col='aki_label_6h',
)

print("\nSaved train / val / test / site_holdout → ../data/processed/")

  train           20906 patients  |  1673348 rows  |  10.4% AKI-positive
  val              4480 patients  |   366823 rows  |  11.1% AKI-positive
  test             4480 patients  |   359603 rows  |  10.4% AKI-positive
  site_holdout        0 patients  |        0 rows  |  AKI label unavailable

Saved train / val / test / site_holdout → ../data/processed/


In [ ]:
!git config --global user.email "daemmyoff1cial@gmail.com"
!git config --global user.name "Sng43"

In [ ]:
%cd /content/drive/MyDrive/sentinel-poc
!git add .
!git commit -m "wip: colab notebook progress before history resync"
!git push

/content/drive/MyDrive/sentinel-poc
[main b8ac1d9] wip: colab notebook progress before history resync
 2 files changed, 2 insertions(+), 500 deletions(-)
 rewrite project_sentinel/notebooks/04_label_engineering.ipynb (87%)
 rewrite project_sentinel/notebooks/05_model_training.ipynb (99%)
Enumerating objects: 11, done.
Counting objects: 100% (11/11), done.
Delta compression using up to 8 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 6.36 KiB | 724.00 KiB/s, done.
Total 6 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 4 local objects.
To https://github.com/Sng43/sentinel-poc.git
   eb83feb..b8ac1d9  main -> main
